## Phase 1 Goals
1. Merge `movies` and `ratings` into one table
2. Remove `(no genres listed)` and `IMAX`
3. Explode genres (one row per genre per movie)
4. Output a clean dataset ready for analysis

---

## Design Decision: Explode vs Fractional Counting

**Goal:** Find genres that are underexposed (high rating score, low rating count).
Rating count = exposure proxy. Total count across genres does NOT need to sum to a fixed number — only relative differences matter.
結論：因為該專案目標是找出被低估（高評價但少評論數量的 movie genre 所以評論總數其實不重要，所以標籤分類採取 explode。
且因為 fractional counting會稀釋掉某些popular genre 的評論數量（因為熱門片東常帶有多種標籤，而多標籤一但被除下來就會被稀釋熱門程度）

**Explode** — each label gets full credit 指的是「當一筆資料同時屬於多個類別」時的標準方法，一筆資料有幾個標籤，就複製成幾筆，並分屬於旗下各個標籤
- A movie with N ratings contributes N to every genre it belongs to
- Example:

| movie_id | genre | rating_count |
|----------|-------|-------------|
| 1 (Dark Knight) | Action | 50,000 |
| 1 | Drama | 50,000 |
| 1 | Sci-Fi | 50,000 |
| 2 (Dandy) | Film-Noir | 3,000 |
| 2 | Drama | 3,000 |

Grouped result:

| genre | total_exposure |
|-------|---------------|
| Action | 50,000 |
| Drama | 53,000 |
| Sci-Fi | 50,000 |
| Film-Noir | 3,000 |

**Fractional Counting** — label credit is split equally 維持資料總數，有多少標籤就除以多少標籤，越多標籤的平均下來各自分配的得數量越少
- A movie with N ratings and K genres contributes N÷K to each genre
- Problem: popular genres (Action, Drama) tend to appear with MORE other genres → get penalized more → gap between popular and niche genres shrinks → harder to detect underexposure

| genre | total_exposure |
|-------|---------------|
| Action | 16,667 |
| Drama | 18,167 |
| Sci-Fi | 16,667 |
| Film-Noir | 1,500 |

**Conclusion: Use Explode.** Fractional counting dilutes popular genres disproportionately and obscures the relative exposure gap we are trying to measure.

---

## Key Concept: Multi-label Data
When one record belongs to multiple categories simultaneously, it is called **multi-label data**.
Explode = copy the row once per label, each copy assigned to one label.

In [1]:
import pandas as pd

movies = pd.read_csv('ml-latest-small/movies.csv')
ratings = pd.read_csv('ml-latest-small/ratings.csv')

print("Loaded movies:", movies.shape)
print("Loaded ratings:", ratings.shape)

Loaded movies: (9742, 3)
Loaded ratings: (100836, 4)


In [3]:
# Merge ratings with movies to get genre info alongside each rating
# We use 'inner' join — only keep movies that exist in BOTH tables

merged = pd.merge(ratings, movies, on='movieId', how='inner') # main chart put right so it appears first
print("Merged shape:", merged.shape)
print("\nFirst few rows:")
print(merged.head())

Merged shape: (100836, 6)

First few rows:
   userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [5]:
# Remove movies with no genre info and IMAX (not a real genre)
cleaned = merged[~merged['genres'].str.contains('no genres listed|IMAX')]

# merge.shape() would give a tuple (xxx, xx) but merged.shape[0] only gives the first figure
print("Before cleaning:", merged.shape[0], "rows")
print("After cleaning:", cleaned.shape[0], "rows")
print("Removed:", merged.shape[0] - cleaned.shape[0], "rows")


Before cleaning: 100836 rows
After cleaning: 96644 rows
Removed: 4192 rows


In [7]:
# Split pipe-separated genres into lists, then explode into individual rows

exploded = cleaned.copy()
exploded['genres'] = exploded['genres'].str.split('|') # 先把 str 變成 list 用 | 分開 >> "Action|Drama|Sci-Fi" 變成 ["Action", "Drama", "Sci-Fi"]
exploded = exploded.explode('genres') #把那個 list 展開成三行，每行一個 genre

print("After explode:", exploded.shape[0], "rows")
print("\nSample:")
print(exploded[exploded['movieId'] == 1][['title', 'genres', 'rating', 'userId']].head(10))#用 movieId==1（Toy Story）來驗證 explode 是否正確，它有 5 個 genre，應該會出現 5 行


After explode: 256281 rows

Sample:
                title     genres  rating  userId
0    Toy Story (1995)  Adventure     4.0       1
0    Toy Story (1995)  Animation     4.0       1
0    Toy Story (1995)   Children     4.0       1
0    Toy Story (1995)     Comedy     4.0       1
0    Toy Story (1995)    Fantasy     4.0       1
516  Toy Story (1995)  Adventure     4.0       5
516  Toy Story (1995)  Animation     4.0       5
516  Toy Story (1995)   Children     4.0       5
516  Toy Story (1995)     Comedy     4.0       5
516  Toy Story (1995)    Fantasy     4.0       5


In [8]:
# Build genre-level summary: average rating (quality) vs rating count (exposure)
# 把所有行按 genre 分組，對每個 group 同時計算兩件事
# agg = aggregate聚合因為平均跟計數都是把「一組數字壓縮成一個數字」
genre_summary = exploded.groupby('genres').agg( 
    avg_rating=('rating', 'mean'), # rating 是欄位、mean, count 是計算方式
    rating_count=('rating', 'count')
).reset_index() #把 groupby 之後的 index 重置，讓結果變成乾淨的 dataframe

genre_summary = genre_summary.sort_values('rating_count', ascending=False) 

# print(dataframe) 如果欄位太多或行數太多，pandas 會自動省略中間的部分，顯示 ...。to_string() 強制把整個 dataframe 完整印出來，不省略任何行。
print(genre_summary.to_string(index=False)) #index=False means不用印genre index



     genres  avg_rating  rating_count
      Drama    3.651830         40868
     Comedy    3.384476         38650
     Action    3.436599         28028
   Thriller    3.491194         25436
  Adventure    3.500139         21623
    Romance    3.504344         17842
      Crime    3.648603         16147
     Sci-Fi    3.447082         15420
    Fantasy    3.475493         10589
   Children    3.391304          8418
    Mystery    3.618348          7347
     Horror    3.256959          7149
  Animation    3.620058          6247
        War    3.815347          4763
    Musical    3.536663          3764
    Western    3.589603          1914
Documentary    3.796849          1206
  Film-Noir    3.920115           870


In [ ]:
genre_summary.to_csv('genre_summary.csv', index=False)
# 因為接下來要在其他檔案中用到genre_summary所以另外儲存成.csv以利呼叫